In [1]:
# 1. 충돌 패키지 제거
!pip uninstall -y -q tensorflow tensorflow-text keras tf-keras tensorflow-estimator tensorflow-decision-forests dopamine-rl

# 2. 버전 고정 설치
!pip install -q tensorflow==2.19.1 tensorflow-text==2.19.0 datasets nltk

In [1]:
import tensorflow as tf

print("TensorFlow 버전:", tf.__version__)
print("GPU 목록:", tf.config.list_physical_devices("GPU"))

TensorFlow 버전: 2.19.1
GPU 목록: []


In [1]:
!nvidia-smi

Fri Apr  3 13:56:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import tensorflow as tf

print("TensorFlow 버전:", tf.__version__)
print("GPU 목록:", tf.config.list_physical_devices("GPU"))

TensorFlow 버전: 2.19.0
GPU 목록: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
import logging
import time
import gc
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_text
import nltk

nltk.download('punkt')

from datasets import load_dataset
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tensorflow.keras.callbacks import EarlyStopping

print("GPU 사용 가능:", tf.config.list_physical_devices("GPU"))
print("TensorFlow 버전:", tf.__version__)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


GPU 사용 가능: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow 버전: 2.19.0


## 데이터셋 로드 및 토크나이저

In [4]:
# opus-100 en-pt: 훈련 1M, 검증 2K, 테스트 2K 쌍
# (우리는 이 중 5K/500만 사용하므로 충분)

raw_ds = load_dataset('Helsinki-NLP/opus-100', 'en-pt')
print('Split별 크기:')
for split in raw_ds:
    print(f'  {split}: {len(raw_ds[split])}')

# 샘플 확인 (opus-100 포맷: {'translation': {'en': ..., 'pt': ...}})
sample = raw_ds['train'][0]
print(f"\n샘플: {sample}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-pt/test-00000-of-00001.parquet:   0%|          | 0.00/220k [00:00<?, ?B/s]

en-pt/train-00000-of-00001.parquet:   0%|          | 0.00/87.2M [00:00<?, ?B/s]

en-pt/validation-00000-of-00001.parquet:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Split별 크기:
  test: 2000
  train: 1000000
  validation: 2000

샘플: {'translation': {'en': 'One, two, three. One, two, three. One, two, three.', 'pt': '1, 2, 3...'}}


### HuggingFace → tf.data.Dataset 변환

In [5]:
# opus-100은 dict 형태이므로, (pt_text, en_text) 튜플로 변환하여
# 기존 파이프라인과 호환되도록 처리

def hf_to_tf_dataset(hf_split):
    """HuggingFace Dataset → tf.data.Dataset (pt, en) 튜플로 변환"""
    pt_texts = [ex['translation']['pt'] for ex in hf_split]
    en_texts = [ex['translation']['en'] for ex in hf_split]
    return tf.data.Dataset.from_tensor_slices((pt_texts, en_texts))

train_examples = hf_to_tf_dataset(raw_ds['train'])
val_examples = hf_to_tf_dataset(raw_ds['validation'])

# 변환 확인
for pt, en in train_examples.take(2):
    print(f'PT: {pt.numpy().decode("utf-8")}')
    print(f'EN: {en.numpy().decode("utf-8")}')
    print()


PT: 1, 2, 3...
EN: One, two, three. One, two, three. One, two, three.

PT: “Kapoc”
EN: Kapok



### 사전 학습된 서브워드 토크나이저 다운로드

In [7]:
# ted_hrlr용 토크나이저를 그대로 사용.
# 이유: 동일한 포르투갈어-영어 도메인이고, 서브워드 토크나이저는
# 범용적이어서 다른 코퍼스에도 잘 작동함.
import os, glob, zipfile

model_name = 'ted_hrlr_translate_pt_en_converter'
zip_path = tf.keras.utils.get_file(
    f'{model_name}.zip',
    f'https://storage.googleapis.com/download.tensorflow.org/models/{model_name}.zip',
    cache_dir='.', cache_subdir=''
)

# 압축 해제 (extract=True가 경로를 예측하기 어려워 수동 처리)
extract_dir = os.path.join(os.path.dirname(zip_path), f'{model_name}_extracted')
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)
    print(f'압축 해제 완료: {extract_dir}')

# saved_model.pb가 있는 정확한 디렉터리 탐색
saved_model_files = glob.glob(
    os.path.join(extract_dir, '**', 'saved_model.pb'), recursive=True
)
if not saved_model_files:
    raise FileNotFoundError(f'{extract_dir} 내에 saved_model.pb를 찾을 수 없습니다')

tokenizer_path = os.path.dirname(saved_model_files[0])
print(f'토크나이저 경로: {tokenizer_path}')

tokenizers = tf.saved_model.load(tokenizer_path)

# 토크나이저 테스트
print("\n포르투갈어 어휘 크기:", tokenizers.pt.get_vocab_size().numpy())
print("영어 어휘 크기:", tokenizers.en.get_vocab_size().numpy())

# opus-100 문장으로 토크나이저 호환성 확인
test_pt = tf.constant(['Eu tenho dois filhos.'])
test_en = tf.constant(['I have two children.'])
print('\nPT 토큰:', tokenizers.pt.tokenize(test_pt).to_list())
print('EN 토큰:', tokenizers.en.tokenize(test_en).to_list())



토크나이저 경로: ./ted_hrlr_translate_pt_en_converter_extracted/ted_hrlr_translate_pt_en_converter

포르투갈어 어휘 크기: 7765
영어 어휘 크기: 7010

PT 토큰: [[2, 104, 225, 220, 554, 16, 3]]
EN 토큰: [[2, 45, 89, 165, 295, 15, 3]]


## 데이터 파이프라인

In [8]:
MAX_TOKENS = 64      # 최대 토큰 수 (이보다 긴 문장은 잘림)
BUFFER_SIZE = 20000  # 셔플 버퍼 크기
BATCH_SIZE = 128     # 배치 크기

def prepare_batch(pt, en):
    """배치 데이터를 모델 입력 형식으로 변환

    포르투갈어 → 인코더 입력
    영어 → 디코더 입력(en_inputs)과 레이블(en_labels)로 분리
    en_inputs: [START] I drank coffee
    en_labels: I drank coffee [END]
    → 한 칸 shift하여 다음 단어를 예측하도록 구성
    """
    pt = tokenizers.pt.tokenize(pt)
    pt = pt[:, :MAX_TOKENS]
    pt = pt.to_tensor()  # ragged → dense (0 패딩)

    en = tokenizers.en.tokenize(en)
    en = en[:, :(MAX_TOKENS + 1)]
    en_inputs = en[:, :-1].to_tensor()   # [END] 제거
    en_labels = en[:, 1:].to_tensor()    # [START] 제거

    return (pt, en_inputs), en_labels


def make_batches(ds, num_samples=5000):
    """tf.data.Dataset을 학습용 배치로 변환"""
    return (
        ds
        .take(num_samples)
        .shuffle(BUFFER_SIZE)
        .batch(BATCH_SIZE)
        .map(prepare_batch, tf.data.AUTOTUNE)
        .prefetch(buffer_size=tf.data.AUTOTUNE)
    )


# 학습/검증 배치 생성
train_batches = make_batches(train_examples, num_samples=5000)
val_batches = make_batches(val_examples, num_samples=500)

# 데이터 형태 확인
for (pt, en), en_labels in train_batches.take(1):
    print(f'포르투갈어 입력 shape: {pt.shape}')
    print(f'영어 입력 shape:      {en.shape}')
    print(f'영어 레이블 shape:    {en_labels.shape}')
    break


포르투갈어 입력 shape: (128, 64)
영어 입력 shape:      (128, 61)
영어 레이블 shape:    (128, 61)


# Transformer 모델 구현

## Positional Encoding
Transformer는 모든 단어를 동시에 처리하므로 순서 정보가 없습니다. sin/cos 함수로 각 위치마다 고유한 벡터를 생성하여 Embedding에 더해줍니다. 짝수 차원에는 sin, 홀수 차원에는 cos을 사용합니다.

In [9]:
def positional_encoding(length, depth):
    """sin/cos 기반 위치 인코딩 생성

    Transformer는 순서 정보가 없으므로, 각 위치에 고유한
    sin/cos 패턴을 더해 순서를 알려줌.

    Args:
        length: 최대 시퀀스 길이
        depth: 임베딩 차원 (d_model)
    Returns:
        shape (length, depth)의 위치 인코딩 텐서
    """
    depth = depth / 2
    positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
    depths = np.arange(depth)[np.newaxis, :] / depth # (1, depth)

    # 주파수 계산: 10000^(2i/d_model)
    angle_rates = 1 / (10000 ** depths)
    angle_rads = positions * angle_rates

    # 짝수 인덱스 = sin, 홀수 인덱스 = cos
    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)], axis=-1)

    return tf.cast(pos_encoding, dtype=tf.float32)


## Positional Embedding Layer
토큰 Embedding과 Positional Encoding을 합치는 레이어입니다. sqrt(d_model)을 곱해 임베딩과 위치 인코딩의 스케일을 맞춥니다.

In [10]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        # 토큰 → 벡터 변환 레이어
        self.embedding = tf.keras.layers.Embedding(
            vocab_size, d_model, mask_zero=True
        )
        # 미리 계산된 위치 인코딩 (최대 2048 토큰)
        self.pos_encoding = positional_encoding(
            length=2048, depth=d_model
        )

    def compute_mask(self, *args, **kwargs):
        return self.embedding.compute_mask(*args, **kwargs)

    def call(self, x):
        length = tf.shape(x)[1]
        x = self.embedding(x)
        # 임베딩 스케일링: sqrt(d_model)을 곱해 위치 인코딩과
        # 스케일을 맞춤 (논문 Section 3.4)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos_encoding[tf.newaxis, :length, :]
        return x


## Attention Layers
Transformer의 3가지 Attention을 모두 구현합니다. 핵심은 return_attention_scores=True로 Head별 가중치를 저장하는 것입니다. 이것이 실험 A(패턴 분석)와 실험 C(Head Pruning)의 기반이 됩니다.

In [11]:
class BaseAttention(tf.keras.layers.Layer):
    """모든 Attention 레이어의 공통 부모 클래스

    Multi-Head Attention + Add & LayerNorm 구조를 공유
    """
    def __init__(self, **kwargs):
        super().__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
        self.layernorm = tf.keras.layers.LayerNormalization()
        self.add = tf.keras.layers.Add()


class GlobalSelfAttention(BaseAttention):
    """Encoder용 Self-Attention

    문장 내 모든 단어가 서로를 참조 가능.
    Q, K, V 모두 같은 입력(x)에서 생성.
    return_attention_scores=True로 Head별 가중치를 저장
    → 실험 A (패턴 분석)에서 활용.
    """
    def call(self, x):
        attn_output, attn_scores = self.mha(
            query=x, value=x, key=x,
            return_attention_scores=True
        )
        self.last_attn_scores = attn_scores
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x


class CausalSelfAttention(BaseAttention):
    """Decoder용 Self-Attention (Masked)

    미래 토큰을 볼 수 없도록 use_causal_mask=True 적용.
    학습 시 치팅 방지: 'I drank' 시점에서 'coffee'를 참조 불가.
    """
    def call(self, x):
        attn_output, attn_scores = self.mha(
            query=x, value=x, key=x,
            use_causal_mask=True,
            return_attention_scores=True
        )
        self.last_attn_scores = attn_scores
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x


class CrossAttention(BaseAttention):
    """Decoder의 Encoder-Decoder Attention

    Query: Decoder의 출력 (현재 번역 중인 문장)
    Key/Value: Encoder의 출력 (원문 문장)
    → Decoder가 원문의 어느 부분에 집중할지 결정.
    """
    def call(self, x, context):
        attn_output, attn_scores = self.mha(
            query=x, key=context, value=context,
            return_attention_scores=True
        )
        self.last_attn_scores = attn_scores
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x


## Feed Forward Network
Self-Attention이 단어 간 관계를 파악한 후, FFN은 각 단어의 표현을 더 풍부하게 변환합니다. 비유하면, Attention은 팀 회의(정보 공유)이고 FFN은 개인 학습(정보 내재화)입니다.

In [12]:
class FeedForward(tf.keras.layers.Layer):
    """Position-wise Feed Forward Network

    구조: Linear(d_model→dff) + ReLU + Linear(dff→d_model) + Dropout

    Self-Attention이 단어 간 관계를 파악했다면,
    FFN은 각 단어의 표현을 더 풍부하게 변환하는 역할.
    비유: Attention = 회의, FFN = 개인 학습
    """
    def __init__(self, d_model, dff, dropout_rate=0.1):
        super().__init__()
        self.seq = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model),
            tf.keras.layers.Dropout(dropout_rate)
        ])
        self.add = tf.keras.layers.Add()
        self.layer_norm = tf.keras.layers.LayerNormalization()

    def call(self, x):
        x = self.add([x, self.seq(x)])  # 잔차 연결
        x = self.layer_norm(x)
        return x


## Encoder
EncoderLayer(GlobalSelfAttention + FFN)를 num_layers개 쌓은 구조입니다. 각 서브층에는 잔차 연결(Residual Connection)과 Layer Normalization이 포함됩니다.

In [13]:
class EncoderLayer(tf.keras.layers.Layer):
    """Encoder의 단일 레이어

    구조: GlobalSelfAttention → FeedForward
    각 서브층에는 잔차 연결 + LayerNorm이 포함됨
    (BaseAttention과 FeedForward 내부에 구현)
    """
    def __init__(self, *, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.self_attention = GlobalSelfAttention(
            num_heads=num_heads, key_dim=d_model // num_heads,
            dropout=dropout_rate
        )
        self.ffn = FeedForward(d_model, dff, dropout_rate)

    def call(self, x):
        x = self.self_attention(x)
        x = self.ffn(x)
        return x


class Encoder(tf.keras.layers.Layer):
    """완전한 Encoder

    구조: PositionalEmbedding + Dropout + EncoderLayer x N
    num_layers개의 EncoderLayer를 쌓아 점점 더 추상적인 표현을 학습
    """
    def __init__(self, *, num_layers, d_model, num_heads,
                 dff, vocab_size, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.pos_embedding = PositionalEmbedding(
            vocab_size=vocab_size, d_model=d_model
        )
        self.enc_layers = [
            EncoderLayer(
                d_model=d_model, num_heads=num_heads,
                dff=dff, dropout_rate=dropout_rate
            )
            for _ in range(num_layers)
        ]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x):
        x = self.pos_embedding(x)
        x = self.dropout(x)
        for enc_layer in self.enc_layers:
            x = enc_layer(x)
        return x


## Decoder
DecoderLayer는 Encoder보다 CrossAttention이 추가되어 3개의 서브층으로 구성됩니다: CausalSelfAttention → CrossAttention → FFN.

In [14]:
class DecoderLayer(tf.keras.layers.Layer):
    """Decoder의 단일 레이어

    구조: CausalSelfAttention → CrossAttention → FeedForward
    Encoder보다 CrossAttention이 추가됨
    """
    def __init__(self, *, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.causal_self_attention = CausalSelfAttention(
            num_heads=num_heads, key_dim=d_model // num_heads,
            dropout=dropout_rate
        )
        self.cross_attention = CrossAttention(
            num_heads=num_heads, key_dim=d_model // num_heads,
            dropout=dropout_rate
        )
        self.ffn = FeedForward(d_model, dff, dropout_rate)

    def call(self, x, context):
        x = self.causal_self_attention(x)
        x = self.cross_attention(x, context)
        self.last_attn_scores = self.cross_attention.last_attn_scores
        x = self.ffn(x)
        return x


class Decoder(tf.keras.layers.Layer):
    """완전한 Decoder"""
    def __init__(self, *, num_layers, d_model, num_heads,
                 dff, vocab_size, dropout_rate=0.1):
        super(Decoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.pos_embedding = PositionalEmbedding(
            vocab_size=vocab_size, d_model=d_model
        )
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.dec_layers = [
            DecoderLayer(
                d_model=d_model, num_heads=num_heads,
                dff=dff, dropout_rate=dropout_rate
            )
            for _ in range(num_layers)
        ]
        self.last_attn_scores = None

    def call(self, x, context):
        x = self.pos_embedding(x)
        x = self.dropout(x)
        for dec_layer in self.dec_layers:
            x = dec_layer(x, context)
        self.last_attn_scores = \
            self.dec_layers[-1].last_attn_scores
        return x


## Transformer 모델
Encoder + Decoder + Final Dense Layer를 조립합니다. Final Dense는 Decoder 출력을 어휘 크기로 변환하여 다음 단어의 확률 분포를 생성합니다.

In [15]:
class Transformer(tf.keras.Model):
    """완전한 Transformer 모델

    구조: Encoder + Decoder + Final Dense Layer
    Final Dense: Decoder 출력을 어휘 크기로 변환하여
    각 위치에서 다음 단어의 확률 분포를 생성
    """
    def __init__(self, *, num_layers, d_model, num_heads, dff,
                 input_vocab_size, target_vocab_size, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(
            num_layers=num_layers, d_model=d_model,
            num_heads=num_heads, dff=dff,
            vocab_size=input_vocab_size, dropout_rate=dropout_rate
        )
        self.decoder = Decoder(
            num_layers=num_layers, d_model=d_model,
            num_heads=num_heads, dff=dff,
            vocab_size=target_vocab_size, dropout_rate=dropout_rate
        )
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inputs):
        context, x = inputs
        context = self.encoder(context)
        x = self.decoder(x, context)
        logits = self.final_layer(x)
        try:
            del logits._keras_mask
        except AttributeError:
            pass
        return logits

# 학습 설정


## Loss, Metric, Learning Rate

패딩 토큰(0)을 무시하는 masked loss/accuracy와 Transformer 논문의 warmup learning rate schedule을 구현합니다.

In [16]:
def masked_loss(label, pred):
    """패딩 토큰(0)을 무시하는 손실 함수

    패딩된 위치의 loss를 0으로 만들어
    실제 토큰에 대해서만 학습이 이루어지도록 함
    """
    mask = label != 0
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction='none'
    )
    loss = loss_object(label, pred)
    mask = tf.cast(mask, dtype=loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)


def masked_accuracy(label, pred):
    """패딩 토큰을 무시하는 정확도"""
    pred = tf.argmax(pred, axis=2)
    label = tf.cast(label, pred.dtype)
    match = label == pred
    mask = label != 0
    match = match & mask
    match = tf.cast(match, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    return tf.reduce_sum(match) / tf.reduce_sum(mask)


class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Transformer 논문의 학습률 스케줄

    warmup 구간에서는 선형 증가, 이후 sqrt 감소.
    공식: lr = d_model^(-0.5) * min(step^(-0.5), step * warmup^(-1.5))
    """
    def __init__(self, d_model, warmup_steps=2000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, dtype=tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

## 번역기
학습된 모델로 실제 번역을 수행하는 클래스입니다. Auto-regressive 방식으로 <START>부터 시작하여 한 단어씩 생성합니다.

In [17]:
class Translator(tf.Module):
    """학습된 Transformer로 문장을 번역하는 클래스

    Auto-regressive 방식: <START> 토큰부터 시작하여
    한 단어씩 생성하며 <END>가 나올 때까지 반복.
    """
    def __init__(self, tokenizers, transformer):
        self.tokenizers = tokenizers
        self.transformer = transformer

    def __call__(self, sentence, max_length=MAX_TOKENS):
        assert isinstance(sentence, tf.Tensor)
        if len(sentence.shape) == 0:
            sentence = sentence[tf.newaxis]

        sentence = self.tokenizers.pt.tokenize(sentence).to_tensor()
        encoder_input = sentence

        # 디코더 시작 토큰: [START]
        start_end = self.tokenizers.en.tokenize([''])[0]
        start = start_end[0][tf.newaxis]
        end = start_end[1][tf.newaxis]

        output_array = tf.TensorArray(
            dtype=tf.int64, size=0, dynamic_size=True
        )
        output_array = output_array.write(0, start)

        for i in tf.range(max_length):
            output = tf.transpose(output_array.stack())
            predictions = self.transformer(
                (encoder_input, output), training=False
            )
            predictions = predictions[:, -1:, :]
            predicted_id = tf.argmax(predictions, axis=-1)
            output_array = output_array.write(i + 1, predicted_id[0])
            if predicted_id == end:
                break

        output = tf.transpose(output_array.stack())
        text = self.tokenizers.en.detokenize(output)[0]
        tokens = self.tokenizers.en.lookup(output)[0]

        self.transformer((encoder_input, output[:, :-1]),
                         training=False)
        attn_weights = self.transformer.decoder.last_attn_scores

        return text, tokens, attn_weights

## 학습 자동화 함수
실험 B에서 num_heads만 바꿔가며 반복 호출할 수 있도록 학습 전체를 함수로 캡슐화합니다. 공정한 비교를 위해 seed와 기타 설정은 모두 고정됩니다.

In [23]:
def train_transformer(num_heads, d_model=96, num_layers=3, dff=384,
                      epochs=30, seed=42):
    """주어진 하이퍼파라미터로 Transformer를 학습하고 결과를 반환

    실험 B에서 num_heads만 바꿔가며 반복 호출.
    공정한 비교를 위해 seed, 데이터, 기타 설정은 모두 고정.

    Args:
        num_heads: 어텐션 헤드 수 (실험 변수)
        d_model: 임베딩 차원 (고정: 96)
        num_layers: 인코더/디코더 레이어 수 (고정: 3)
        dff: FFN 은닉층 크기 (고정: 384)
        epochs: 최대 학습 에폭 (고정: 30)
        seed: 랜덤 시드 (고정: 42)
    Returns:
        (transformer, history, elapsed_time)
    """
    tf.random.set_seed(seed)
    np.random.seed(seed)

    # 모델 생성
    transformer = Transformer(
        num_layers=num_layers,
        d_model=d_model,
        num_heads=num_heads,
        dff=dff,
        input_vocab_size=int(tokenizers.pt.get_vocab_size().numpy()),
        target_vocab_size=int(tokenizers.en.get_vocab_size().numpy()),
        dropout_rate=0.1
    )

    # 옵티마이저 설정
    lr_schedule = CustomSchedule(d_model)
    optimizer = tf.keras.optimizers.Adam(
        lr_schedule, beta_1=0.9, beta_2=0.98, epsilon=1e-9
    )

    transformer.compile(
        loss=masked_loss,
        optimizer=optimizer,
        metrics=[masked_accuracy]
    )

    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=10,
            restore_best_weights=True, verbose=1
        )
    ]

    start_time = time.time()

    history = transformer.fit(
        train_batches,
        epochs=epochs,
        validation_data=val_batches,
        callbacks=callbacks,
        verbose=1
    )

    elapsed = time.time() - start_time
    print(f'\n학습 완료! 소요 시간: {elapsed:.1f}초')
    print(f'Head 수: {num_heads}, head_dim: {d_model // num_heads}')

    return transformer, history, elapsed

## BLEU Score 평가 함수
BLEU(Bilingual Evaluation Understudy)는 기계 번역의 표준 평가 지표입니다. n-gram precision을 기반으로 모델 번역과 정답 번역의 유사도를 0~1 범위로 측정합니다.

In [24]:
# 테스트 문장 세트 (다양한 구문 구조를 포함)
TEST_SENTENCES = [
    ('este e um problema que temos que resolver.',
     'this is a problem we have to solve .'),
    ('os meus vizinhos ouviram sobre esta ideia.',
     'and my neighboring homes heard about this idea .'),
    ('este e o primeiro livro que eu fiz.',
     "this is the first book i've ever done."),
    ('eu tenho dois filhos.',
     'i have two children.'),
    ('obrigado por estarem aqui.',
     'thank you for being here.'),
    ('a vida e muito curta.',
     'life is very short.'),
    ('eu nao sei o que fazer.',
     'i do not know what to do.'),
    ('este e um dia muito importante.',
     'this is a very important day.'),
    ('eu quero mudar o mundo.',
     'i want to change the world.'),
    ('as pessoas precisam de ajuda.',
     'people need help.'),
]


def evaluate_bleu(transformer, test_sentences=TEST_SENTENCES):
    """테스트 문장들에 대한 BLEU Score 계산

    BLEU (Bilingual Evaluation Understudy):
    기계 번역 품질의 표준 지표. 0~1 범위이며 1에 가까울수록 좋음.
    n-gram precision을 기반으로 정답과의 유사도를 측정.
    """
    translator = Translator(tokenizers, transformer)
    bleu_scores = []
    smoothie = SmoothingFunction().method1

    print(f"{'='*60}")
    print(f"{'원문':<30} | {'번역':<25} | BLEU")
    print(f"{'='*60}")

    for pt_sent, en_ref in test_sentences:
        translated, _, _ = translator(
            tf.constant(pt_sent)
        )
        translated_str = translated.numpy().decode('utf-8')

        # BLEU 계산
        ref_tokens = en_ref.lower().split()
        hyp_tokens = translated_str.lower().split()
        score = sentence_bleu(
            [ref_tokens], hyp_tokens,
            smoothing_function=smoothie
        )
        bleu_scores.append(score)

        print(f"{pt_sent[:28]:<30} | {translated_str[:23]:<25} | {score:.3f}")

    avg_bleu = np.mean(bleu_scores)
    print(f"{'='*60}")
    print(f"평균 BLEU Score: {avg_bleu:.4f}")
    return bleu_scores, avg_bleu

# 실험 B: Head 수 변화에 따른 성능 비교

## 실험 설계
d_model=96을 고정하고 num_heads를 1, 2, 6, 12로 변화시킵니다. head_dim = d_model / num_heads이므로, Head 수가 많아질수록 각 Head의 차원이 줄어듭니다.


In [ ]:
# d_model=96 고정, num_heads만 변경
# 96의 약수: 1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 96
HEAD_CONFIGS = {
    '1 Head (dim=96)': 1,    # 단일 헤드
    '2 Heads (dim=48)': 2,   # 최소 멀티헤드
    '6 Heads (dim=16)': 6,   # 기본 설정
    '12 Heads (dim=8)': 12,  # 과도한 분할
}

# 결과 저장용 딕셔너리
results = {}

for name, n_heads in HEAD_CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'실험: {name}')
    print(f'{"="*60}')

    model, history, elapsed = train_transformer(
        num_heads=n_heads
    )

    # BLEU 평가
    bleu_scores, avg_bleu = evaluate_bleu(model)

    # 결과 저장
    results[name] = {
        'num_heads': n_heads,
        'head_dim': 96 // n_heads,
        'history': history.history,
        'avg_bleu': avg_bleu,
        'bleu_scores': bleu_scores,
        'train_time': elapsed,
        'best_val_loss': min(history.history['val_loss']),
        'best_val_acc': max(history.history['val_masked_accuracy']),
        'model': model,  # Head pruning 실험용으로 보관
    }

    # 메모리 관리: 모델은 유지하되 GPU 캐시 정리
    gc.collect()

print('\n모든 실험 완료!')


실험: 1 Head (dim=96)
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'global_self_attention_6' (of type GlobalSelfAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_6' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'causal_self_attention_6' (of type CausalSelfAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the 

40/40 ━━━━━━━━━━━━━━━━━━━━ 148s 3s/step - loss: 8.8160 - masked_accuracy: 0.0068 - val_loss: 8.7115 - val_masked_accuracy: 0.0390
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - loss: 8.5056 - masked_accuracy: 0.0989 - val_loss: 8.3734 - val_masked_accuracy: 0.0820
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - loss: 8.0790 - masked_accuracy: 0.1285 - val_loss: 7.9081 - val_masked_accuracy: 0.0902
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 61s 2s/step - loss: 7.5192 - masked_accuracy: 0.1328 - val_loss: 7.3533 - val_masked_accuracy: 0.0912
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 16s 388ms/step - loss: 6.9141 - masked_accuracy: 0.1348 - val_loss: 6.8316 - val_masked_accuracy: 0.0918
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - loss: 6.3870 - masked_accuracy: 0.1370 - val_loss: 6.4699 - val_masked_accuracy: 0.0938
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 31s 786ms/step - loss: 6.0221 - masked_accuracy: 0.1511 - val_loss: 6.2489 - val_masked_accuracy: 0.1201
Epoch 8/30
40/40 ━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (1, 1, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


este e um problema que temos   | you ' s a little s a li   | 0.025
os meus vizinhos ouviram sob   | i ' m sorry to be a lit   | 0.024
este e o primeiro livro que    | i ' s a .                 | 0.000
eu tenho dois filhos.          | i ' m sorry to be a lot   | 0.019
obrigado por estarem aqui.     | i ' m sorry .             | 0.000
a vida e muito curta.          | i ' m sorry .             | 0.000
eu nao sei o que fazer.        | i ' m not you .           | 0.041
este e um dia muito importan   | i ' m a good with a lit   | 0.003
eu quero mudar o mundo.        | i ' m a sgcco .           | 0.041
as pessoas precisam de ajuda   | i ' m sorry , i ' m sor   | 0.000
평균 BLEU Score: 0.0153

실험: 2 Heads (dim=48)
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'global_self_attention_9' (of type GlobalSelfAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_9' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'causal_self_attention_9' (of type CausalSelfAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the 

40/40 ━━━━━━━━━━━━━━━━━━━━ 194s 4s/step - loss: 8.8266 - masked_accuracy: 0.0069 - val_loss: 8.7285 - val_masked_accuracy: 0.0457
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step - loss: 8.5369 - masked_accuracy: 0.0806 - val_loss: 8.4086 - val_masked_accuracy: 0.0510
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - loss: 8.1189 - masked_accuracy: 0.0816 - val_loss: 7.9487 - val_masked_accuracy: 0.0524
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - loss: 7.5593 - masked_accuracy: 0.0832 - val_loss: 7.3918 - val_masked_accuracy: 0.0537
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - loss: 6.9488 - masked_accuracy: 0.0975 - val_loss: 6.8645 - val_masked_accuracy: 0.0756
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step - loss: 6.4137 - masked_accuracy: 0.1355 - val_loss: 6.4924 - val_masked_accuracy: 0.0884
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - loss: 6.0325 - masked_accuracy: 0.1645 - val_loss: 6.2577 - val_masked_accuracy: 0.1191
Epoch 8/30
40/40 ━━━━━━━━━━━━

## 결과 시각화 코드

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Attention Head 수에 따른 성능 비교',
             fontsize=16, fontweight='bold')

colors = ['#e74c3c', '#3498db', '#27ae60', '#f39c12']

# (1) 학습 손실 곡선
ax = axes[0, 0]
for i, (name, r) in enumerate(results.items()):
    ax.plot(r['history']['loss'], color=colors[i],
            label=name, linewidth=1.5)
ax.set_title('Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (2) 검증 손실 곡선
ax = axes[0, 1]
for i, (name, r) in enumerate(results.items()):
    ax.plot(r['history']['val_loss'], color=colors[i],
            label=name, linewidth=1.5)
ax.set_title('Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (3) BLEU Score 비교 (막대 그래프)
ax = axes[1, 0]
names = list(results.keys())
bleus = [results[n]['avg_bleu'] for n in names]
bars = ax.bar(range(len(names)), bleus, color=colors)
ax.set_title('Average BLEU Score')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split('(')[0].strip() for n in names],
                    fontsize=9)
ax.set_ylabel('BLEU')
for bar, val in zip(bars, bleus):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# (4) 학습 시간 비교
ax = axes[1, 1]
times = [results[n]['train_time'] / 60 for n in names]
bars = ax.bar(range(len(names)), times, color=colors)
ax.set_title('Training Time (minutes)')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split('(')[0].strip() for n in names],
                    fontsize=9)
ax.set_ylabel('Minutes')
for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('exp_b_results.png', dpi=150, bbox_inches='tight')
plt.show()

# 요약 테이블 출력
print(f"\n{'설정':<22} | {'val_loss':>10} | {'val_acc':>10} | {'BLEU':>8} | {'시간(분)':>8}")
print('-' * 70)
for name, r in results.items():
    print(f"{name:<22} | {r['best_val_loss']:>10.4f} | "
          f"{r['best_val_acc']:>10.4f} | {r['avg_bleu']:>8.4f} | "
          f"{r['train_time']/60:>8.1f}")


# Attention Head 패턴 시각화
학습된 Transformer의 각 Attention Head가 실제로 어떤 패턴을 포착하는지 heatmap으로 시각화합니다. 주요 패턴 유형:
패턴	설명	의미
대각선	자기 자신에 집중	positional/identity 정보
이전/다음 토큰	인접 단어에 집중	어순 관계
특정 토큰 집중	[START]/[END] 등에 집중	문장 경계 인식
분산	여러 토큰에 고르게 분포	문맥 전체 참조


## 시각화 코드

In [ ]:
# 기본 설정(6 Heads) 모델 사용
base_model = results['6 Heads (dim=16)']['model']

def visualize_encoder_heads(model, sentence, tokenizers, layer_idx=0):
    """특정 Encoder layer의 모든 Attention Head를 시각화

    각 Head가 문장 내 어떤 단어 쌍에 높은 가중치를 부여하는지
    heatmap으로 표시. 다른 패턴을 보이면 다른 역할을 학습한 것.
    """
    # 1) 토큰화
    in_tokens = tokenizers.pt.tokenize(
        tf.constant([sentence])
    ).to_tensor()

    # 2) Encoder 통과 → attention score 자동 저장
    _ = model.encoder(in_tokens, training=False)

    # 3) 해당 layer의 attention scores 추출
    attn = model.encoder.enc_layers[layer_idx] \
        .self_attention.last_attn_scores
    attn = tf.squeeze(attn, 0).numpy()  # (num_heads, seq, seq)

    # 4) 토큰 텍스트 변환
    token_labels = tokenizers.pt.lookup(in_tokens)[0].numpy()
    token_labels = [t.decode('utf-8') for t in token_labels]

    # 5) 시각화
    n_heads = attn.shape[0]
    cols = min(n_heads, 3)
    rows = (n_heads + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    if n_heads == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for h in range(n_heads):
        ax = axes[h]
        im = ax.matshow(attn[h], cmap='viridis', vmin=0, vmax=1)
        ax.set_title(f'Head {h}', fontsize=12, fontweight='bold')
        ax.set_xticks(range(len(token_labels)))
        ax.set_xticklabels(token_labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(token_labels)))
        ax.set_yticklabels(token_labels, fontsize=7)

    # 빈 subplot 숨기기
    for h in range(n_heads, len(axes)):
        axes[h].set_visible(False)

    fig.suptitle(f'Encoder Layer {layer_idx} - Attention Patterns',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'attention_layer{layer_idx}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    return attn


# 테스트 문장으로 각 Encoder layer 시각화
test_sent = 'este e um problema que temos que resolver.'
print(f'분석 문장: "{test_sent}"\n')

all_attn = {}
for layer_idx in range(3):  # num_layers = 3
    print(f'--- Encoder Layer {layer_idx} ---')
    all_attn[layer_idx] = visualize_encoder_heads(
        base_model, test_sent, tokenizers, layer_idx
    )

## 엔트로피 및 Head 유사도 분석
Attention entropy는 각 Head가 "얼마나 분산적으로 주목하는가"를 정량화합니다. Head 간 코사인 유사도는 중복된 Head를 찾는 데 사용됩니다.
💡 Entropy 높음 → 분산 패턴 (여러 토큰에 고르게), Entropy 낮음 → 집중 패턴 (특정 토큰에 몰림)


In [ ]:

def attention_entropy(attn_weights):
    """Attention 분포의 엔트로피 계산

    엔트로피가 높으면 → 여러 토큰에 고르게 주목 (분산 패턴)
    엔트로피가 낮으면 → 특정 토큰에 집중 (집중 패턴)

    정보이론에서 엔트로피 = -sum(p * log(p))
    최대 엔트로피 = log(seq_len) (균등 분포일 때)
    """
    # attn_weights: (num_heads, seq_len, seq_len)
    eps = 1e-10
    entropy = -np.sum(
        attn_weights * np.log(attn_weights + eps), axis=-1
    )
    # 각 head의 평균 엔트로피
    return entropy.mean(axis=-1)  # (num_heads,)


def head_similarity(attn_weights):
    """Head 간 코사인 유사도 계산

    유사도가 높은 Head 쌍은 비슷한 패턴을 학습한 것
    → 하나를 제거해도 성능 저하가 적을 가능성
    """
    n_heads = attn_weights.shape[0]
    # 각 head의 attention을 1D 벡터로 변환
    flat = attn_weights.reshape(n_heads, -1)
    # 정규화
    norms = np.linalg.norm(flat, axis=1, keepdims=True) + 1e-10
    flat_norm = flat / norms
    # 코사인 유사도 행렬
    sim_matrix = flat_norm @ flat_norm.T
    return sim_matrix


# 엔트로피 분석
print('=== Attention Entropy 분석 ===')
print('(높을수록 분산 패턴, 낮을수록 집중 패턴)\n')

for layer_idx in range(3):
    entropy = attention_entropy(all_attn[layer_idx])
    print(f'Encoder Layer {layer_idx}:')
    for h, e in enumerate(entropy):
        pattern = '분산' if e > np.median(entropy) else '집중'
        print(f'  Head {h}: entropy = {e:.4f} ({pattern} 패턴)')
    print()

# Head 유사도 분석
print('\n=== Head 간 코사인 유사도 (Layer 0) ===')
sim = head_similarity(all_attn[0])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.matshow(sim, cmap='RdYlBu_r', vmin=0, vmax=1)
ax.set_title('Head Similarity (Layer 0)', fontweight='bold')
ax.set_xlabel('Head')
ax.set_ylabel('Head')
plt.colorbar(im)
for i in range(sim.shape[0]):
    for j in range(sim.shape[1]):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='black' if sim[i,j] < 0.7 else 'white')
plt.savefig('head_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

# 유사한 Head 쌍 찾기
print('\n유사도가 높은 Head 쌍 (>0.8):')
for i in range(sim.shape[0]):
    for j in range(i+1, sim.shape[1]):
        if sim[i, j] > 0.8:
            print(f'  Head {i} - Head {j}: {sim[i,j]:.3f}')


# Head Pruning
학습 완료된 6-Head 모델에서 각 Head를 하나씩 0으로 마스킹한 후 val_loss 변화를 측정합니다. val_loss 변화가 작은 Head는 제거해도 성능 영향이 적은 "불필요한 Head"입니다.
구체적으로, Multi-Head Attention의 output projection 가중치에서 특정 Head에 해당하는 행(row)을 0으로 설정합니다. 이렇게 하면 모델을 재학습하지 않고도 각 Head의 중요도를 빠르게 측정할 수 있습니다.


## Head Pruning 코드

In [ ]:
def evaluate_model(model, val_batches):
    """모델의 검증 손실과 정확도를 반환"""
    results = model.evaluate(val_batches, verbose=0)
    return {'val_loss': results[0], 'val_acc': results[1]}


def mask_head(model, layer_idx, head_idx, num_heads, d_model):
    """특정 Encoder layer의 특정 Head를 0으로 마스킹

    원리:
    Multi-Head Attention의 output projection 가중치에서
    특정 head에 해당하는 행(row)을 0으로 만듦.
    → 해당 head의 기여가 0이 되어 마치 head가 없는 것처럼 동작.

    이렇게 하면 모델을 재학습하지 않고도
    각 head의 중요도를 빠르게 측정할 수 있음.
    """
    enc_layer = model.encoder.enc_layers[layer_idx]
    dense = enc_layer.self_attention.mha._output_dense
    weights = dense.get_weights()  # [kernel, bias]
    head_dim = d_model // num_heads
    start = head_idx * head_dim
    end = start + head_dim
    weights[0][start:end, :] = 0  # 해당 head의 kernel을 0으로
    dense.set_weights(weights)


def restore_weights(model, original_weights_dict):
    """마스킹 전 원래 가중치로 복원"""
    for (layer_idx, head_idx), w in original_weights_dict.items():
        enc_layer = model.encoder.enc_layers[layer_idx]
        dense = enc_layer.self_attention.mha._output_dense
        dense.set_weights(w)


# 기본 모델 (6 Heads) 사용
model = results['6 Heads (dim=16)']['model']
NUM_HEADS = 6
D_MODEL = 96
NUM_LAYERS = 3

# 원래 성능 측정
baseline = evaluate_model(model, val_batches)
print(f"Baseline - val_loss: {baseline['val_loss']:.4f}, "
      f"val_acc: {baseline['val_acc']:.4f}")

# 각 Head를 개별적으로 마스킹하여 중요도 측정
print('\n=== Head별 중요도 측정 (마스킹 후 val_loss 변화) ===')
importance = {}

for layer_idx in range(NUM_LAYERS):
    print(f'\n--- Encoder Layer {layer_idx} ---')
    for head_idx in range(NUM_HEADS):
        # 원래 가중치 저장
        enc_layer = model.encoder.enc_layers[layer_idx]
        dense = enc_layer.self_attention.mha._output_dense
        original_w = [w.copy() for w in dense.get_weights()]

        # 마스킹
        mask_head(model, layer_idx, head_idx, NUM_HEADS, D_MODEL)

        # 성능 측정
        masked_result = evaluate_model(model, val_batches)
        loss_increase = masked_result['val_loss'] - baseline['val_loss']
        importance[(layer_idx, head_idx)] = loss_increase

        print(f'  Head {head_idx}: val_loss 변화 = +{loss_increase:.4f}')

        # 가중치 복원
        dense.set_weights(original_w)


## 결과 시각화 및 누적 Pruning

개별 Head 중요도 히트맵과, 중요도 낮은 순으로 누적 제거하며 성능 변화를 추적하는 그래프를 생성합니다.

In [ ]:
# 1) Head 중요도 히트맵
imp_matrix = np.zeros((NUM_LAYERS, NUM_HEADS))
for (l, h), val in importance.items():
    imp_matrix[l, h] = val

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.matshow(imp_matrix, cmap='YlOrRd')
ax.set_title('Head Importance\n(val_loss increase when masked)',
             fontweight='bold', pad=20)
ax.set_xlabel('Head Index')
ax.set_ylabel('Encoder Layer')
ax.set_xticks(range(NUM_HEADS))
ax.set_yticks(range(NUM_LAYERS))
plt.colorbar(im, ax=ax)
for i in range(NUM_LAYERS):
    for j in range(NUM_HEADS):
        ax.text(j, i, f'{imp_matrix[i,j]:.3f}',
                ha='center', va='center', fontsize=9)

# 2) 누적 pruning 실험
# 중요도가 낮은 순으로 정렬
sorted_heads = sorted(importance.items(), key=lambda x: x[1])

print('Head 중요도 순위 (낮은순 = 제거 우선):')
for rank, ((l, h), imp) in enumerate(sorted_heads):
    print(f'  {rank+1}. Layer {l} Head {h}: +{imp:.4f}')

# 하나씩 누적 제거하며 성능 변화 추적
# 먼저 모든 원래 가중치를 저장
all_original_weights = {}
for layer_idx in range(NUM_LAYERS):
    for head_idx in range(NUM_HEADS):
        enc_layer = model.encoder.enc_layers[layer_idx]
        dense = enc_layer.self_attention.mha._output_dense
        all_original_weights[(layer_idx, head_idx)] = \
            [w.copy() for w in dense.get_weights()]

cumulative_losses = [baseline['val_loss']]
cumulative_accs = [baseline['val_acc']]
pruned_count = [0]

# 가중치를 원래대로 복원
for layer_idx in range(NUM_LAYERS):
    enc_layer = model.encoder.enc_layers[layer_idx]
    dense = enc_layer.self_attention.mha._output_dense
    dense.set_weights(all_original_weights[(layer_idx, 0)])

# 재복원 (전체)
for (l, h), w in all_original_weights.items():
    enc_layer = model.encoder.enc_layers[l]
    dense = enc_layer.self_attention.mha._output_dense
    dense.set_weights(w)

# 누적 제거
for i, ((l, h), _) in enumerate(sorted_heads):
    mask_head(model, l, h, NUM_HEADS, D_MODEL)
    result = evaluate_model(model, val_batches)
    cumulative_losses.append(result['val_loss'])
    cumulative_accs.append(result['val_acc'])
    pruned_count.append(i + 1)
    print(f'Pruned {i+1}/{NUM_HEADS*NUM_LAYERS} heads: '
          f"val_loss={result['val_loss']:.4f}, "
          f"val_acc={result['val_acc']:.4f}")

# 가중치 복원
for (l, h), w in all_original_weights.items():
    enc_layer = model.encoder.enc_layers[l]
    dense = enc_layer.self_attention.mha._output_dense
    dense.set_weights(w)

# 누적 pruning 그래프
ax = axes[1]
total_heads = NUM_HEADS * NUM_LAYERS
x_pct = [n / total_heads * 100 for n in pruned_count]
ax.plot(x_pct, cumulative_losses, 'o-', color='#e74c3c',
        linewidth=2, markersize=4, label='val_loss')
ax.axhline(y=baseline['val_loss'], color='gray',
           linestyle='--', alpha=0.7, label='Baseline')
ax.set_title('Cumulative Head Pruning\n(least important first)',
             fontweight='bold')
ax.set_xlabel('Heads Pruned (%)')
ax.set_ylabel('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('head_pruning_results.png', dpi=150, bbox_inches='tight')
plt.show()


# 결과 저장 및 정리

In [ ]:
# JSON으로 결과 저장 (GitHub용)
summary = {}
for name, r in results.items():
    summary[name] = {
        'num_heads': r['num_heads'],
        'head_dim': r['head_dim'],
        'best_val_loss': float(r['best_val_loss']),
        'best_val_acc': float(r['best_val_acc']),
        'avg_bleu': float(r['avg_bleu']),
        'train_time_sec': float(r['train_time']),
    }

with open('experiment_results.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('결과가 experiment_results.json에 저장되었습니다.')
print('\n' + '='*60)
print('실험 완료! 다음 파일들이 생성되었습니다:')
print('  - attention_layer0~2.png   (실험 A: Head 패턴)')
print('  - head_similarity.png      (실험 A: Head 유사도)')
print('  - exp_b_results.png        (실험 B: 성능 비교)')
print('  - head_pruning_results.png (실험 C: Pruning 결과)')
print('  - experiment_results.json  (수치 결과)')
print('='*60)
